# Example 0.13.2 / 2: Exercise 6-5: The Diet Model with a Set of Pairs

Companion notebook for **0.13.2 Exercises on Sets, Parameters and Variables** in the AMPL lab Documentation, section "Exercise 6-5: The Diet Model with a Set of Pairs". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 6-5: The Diet Model with a Set of Pairs

Exercise 6-5(a) reformulates the diet model with a set `GIVE within {NUTR,FOOD}` of the pairs
$(i,j)$ for which food $j$ contains nutrient $i$. Only those pairs need data, and the constraint
sums over the foods paired with each nutrient (Sets of Pairs (AMPL lab documentation, section 4.5)). This version uses the data
of Exercise 2-1 on Exercises on Building Models (AMPL lab documentation, section 0.13.1), leaving out the four zero entries.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NUTR;                         # nutrients
set FOOD;                         # foods
set GIVE within {NUTR, FOOD};     # (i,j): food j contains nutrient i
param cost {FOOD} > 0;            # dollars per pound
param amt {GIVE} > 0;             # nutrient units per pound, only for pairs in GIVE
param required {NUTR} >= 0;       # daily requirement

var Buy {FOOD} >= 0;              # pounds bought

minimize Total_Cost: sum {j in FOOD} cost[j] * Buy[j];  # dollars per day

# sum only over the foods that contain nutrient i
subject to Need {i in NUTR}: sum {(i,j) in GIVE} amt[i,j] * Buy[j] >= required[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["NUTR"] = ["calories", "protein", "calcium", "vitaminA"]
ampl.set["FOOD"] = ["bread", "meat", "potatoes", "cabbage", "milk", "gelatin"]
ampl.param["cost"] = {"bread": 0.30, "meat": 1.00, "potatoes": 0.05,
                      "cabbage": 0.08, "milk": 0.23, "gelatin": 0.48}
amt = {
    ("calories", "bread"): 1254, ("calories", "meat"): 1457, ("calories", "potatoes"): 318,
    ("calories", "cabbage"): 46, ("calories", "milk"): 309, ("calories", "gelatin"): 1725,
    ("protein", "bread"): 39, ("protein", "meat"): 73, ("protein", "potatoes"): 8,
    ("protein", "cabbage"): 4, ("protein", "milk"): 16, ("protein", "gelatin"): 43,
    ("calcium", "bread"): 418, ("calcium", "meat"): 41, ("calcium", "potatoes"): 42,
    ("calcium", "cabbage"): 141, ("calcium", "milk"): 536,
    ("vitaminA", "potatoes"): 70, ("vitaminA", "cabbage"): 860, ("vitaminA", "milk"): 720,
}
ampl.set["GIVE"] = list(amt)
ampl.param["amt"] = amt
ampl.param["required"] = {"calories": 3000, "protein": 70, "calcium": 800, "vitaminA": 500}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, round(ampl.obj["Total_Cost"].value(), 4))
print("pairs:", ampl.get_value("card(GIVE)"), "of", ampl.get_value("card(NUTR) * card(FOOD)"))

HiGHS 1.14.0: optimal solution; objective 0.6196227853
2 simplex iterations
0 barrier iterations
solved 0.6196
pairs: 20 of 24


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set NUTR := calories protein calcium vitaminA;
set FOOD := bread meat potatoes cabbage milk gelatin;
param cost := bread 0.30  meat 1.00  potatoes 0.05  cabbage 0.08  milk 0.23  gelatin 0.48;
param: GIVE: amt :=
calories bread 1254   calories meat 1457   calories potatoes 318
calories cabbage 46   calories milk 309    calories gelatin 1725
protein bread 39      protein meat 73      protein potatoes 8
protein cabbage 4     protein milk 16      protein gelatin 43
calcium bread 418     calcium meat 41      calcium potatoes 42
calcium cabbage 141   calcium milk 536
vitaminA potatoes 70  vitaminA cabbage 860 vitaminA milk 720;
param required := calories 3000  protein 70  calcium 800  vitaminA 500;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Total_Cost, card(GIVE);

HiGHS 1.14.0: optimal solution; objective 0.6196227853
2 simplex iterations
0 barrier iterations
Total_Cost = 0.619623
card(GIVE) = 20

